# FraudShield AI — Phase 2: Understand the Data

This notebook explores the anonymized credit-card fraud dataset before any model is trained.

**Questions we will answer**
1. What are the rows, columns, data types, and memory footprint?
2. What do `Time`, `Amount`, `V1`–`V28`, and `Class` represent?
3. How severe is class imbalance?
4. Are there missing values or duplicate rows / duplicate feature vectors?
5. How do transaction amount and relative time differ by class?
6. Which numerical features correlate with `Class`, and what can correlation not tell us?

> The raw CSV is never modified in this notebook. Do not remove duplicates or resample the data during EDA; those are decisions to investigate and validate later.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config, resolve_data_path, PROJECT_ROOT

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

config = load_config()
data_path = resolve_data_path(config)
print(f"Dataset path: {data_path}")
if not data_path.is_file():
    raise FileNotFoundError(
        "Dataset not found. Download creditcard.csv from the Kaggle dataset page and place it in data/raw/."
    )

df = pd.read_csv(data_path)
TARGET = config["data"].get("target_column", "Class")
expected_columns = config["data"].get("expected_columns", [])
print("Loaded successfully.")

## 1. Dataset structure

In this dataset, each row represents a transaction. `Class` is the target label: `0` means legitimate and `1` means fraud. `Time` is a relative elapsed-time feature in seconds, not a real calendar timestamp. `V1`–`V28` are anonymized PCA components, so do not invent business meanings for individual components. `Amount` is the transaction amount.

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")
print(f"Memory: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MiB")
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst five rows:")
display(df.head())
print("\nData types:")
display(df.dtypes.rename("dtype").to_frame())
print("\nDescriptive statistics:")
display(df.describe().T)

missing_expected = [c for c in expected_columns if c not in df.columns]
extra_columns = [c for c in df.columns if c not in expected_columns]
print(f"\nMissing expected columns: {missing_expected}")
print(f"Unexpected columns: {extra_columns}")
assert not missing_expected, "The CSV is missing columns defined in configs/config.yaml."
assert TARGET in df.columns, f"Target column {TARGET!r} is missing."

## 2. Understand class imbalance

Class imbalance means one label appears much less often than the other. A model that always predicts the majority class can still obtain deceptively high accuracy, so we will inspect counts and percentages instead of relying on accuracy.

In [ ]:
class_counts = df[TARGET].value_counts(dropna=False).sort_index()
class_percentages = df[TARGET].value_counts(normalize=True, dropna=False).sort_index() * 100
class_table = pd.DataFrame({"count": class_counts, "percentage": class_percentages})
display(class_table)

fig, ax = plt.subplots(figsize=(8, 5))
labels = ["Legitimate (0)" if value == 0 else "Fraud (1)" if value == 1 else f"Class {value}" for value in class_counts.index]
bars = ax.bar(labels, class_counts.values)
ax.set_title("Class imbalance")
ax.set_ylabel("Number of transactions")
for bar, value, pct in zip(bars, class_counts.values, class_percentages.values):
    ax.annotate(f"{value:,}\n{pct:.3f}%", (bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 5), textcoords="offset points", ha="center", fontsize=9)
fig.tight_layout()
plt.show()

fraud_count = int((df[TARGET] == 1).sum())
legitimate_count = int((df[TARGET] == 0).sum())
fraud_pct = fraud_count / len(df) * 100
print(f"Fraud transactions: {fraud_count:,} ({fraud_pct:.4f}%)")
print(f"Legitimate transactions: {legitimate_count:,} ({100 - fraud_pct:.4f}%)")

## 3. Missing values

Count missing values per column and inspect their percentages. A missing value count of zero does not mean the data has no other quality issues; it only means pandas did not identify null values.

In [ ]:
missing_table = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": df.isna().mean() * 100,
}).sort_values("missing_count", ascending=False)
display(missing_table[missing_table["missing_count"] > 0] if (missing_table["missing_count"] > 0).any() else missing_table.head(10))

if (missing_table["missing_count"] > 0).any():
    plot_missing = missing_table[missing_table["missing_count"] > 0].sort_values("missing_count")
    ax = plot_missing["missing_count"].plot(kind="barh", figsize=(9, 5), title="Missing values by column")
    ax.set_xlabel("Missing value count")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found in the loaded CSV.")

## 4. Duplicate detection — do not delete records automatically

We check two things:
- **Exact duplicate rows:** every feature and the class label match.
- **Duplicate feature vectors:** input features match when `Class` is excluded. If the target labels differ inside one such group, the same features have contradictory labels and need investigation.

A duplicate can represent repeated data, but whether it should be removed depends on its origin and the evaluation strategy. This step only reports candidates; it changes nothing.

In [ ]:
feature_columns = [column for column in df.columns if column != TARGET]
exact_duplicate_extra_rows = int(df.duplicated().sum())
feature_duplicate_extra_rows = int(df.duplicated(subset=feature_columns).sum())
feature_duplicate_mask = df.duplicated(subset=feature_columns, keep=False)
rows_in_feature_duplicate_groups = int(feature_duplicate_mask.sum())

conflicting_label_groups = 0
if feature_duplicate_extra_rows:
    duplicate_candidates = df.loc[feature_duplicate_mask, feature_columns + [TARGET]]
    label_counts_per_feature_group = duplicate_candidates.groupby(
        feature_columns, dropna=False, sort=False
    )[TARGET].nunique(dropna=False)
    conflicting_label_groups = int((label_counts_per_feature_group > 1).sum())

duplicate_summary = pd.DataFrame([
    {"check": "exact duplicate extra rows", "count": exact_duplicate_extra_rows},
    {"check": "duplicate input-feature extra rows (Class excluded)", "count": feature_duplicate_extra_rows},
    {"check": "rows in duplicate input-feature groups", "count": rows_in_feature_duplicate_groups},
    {"check": "duplicate feature groups with conflicting Class labels", "count": conflicting_label_groups},
])
display(duplicate_summary)

if exact_duplicate_extra_rows:
    print("Exact duplicate rows exist. Do not drop them until their meaning and effect on splitting are understood.")
if conflicting_label_groups:
    print("WARNING: identical input features appear with conflicting Class labels. Investigate these groups.")

## 5. Compare `Amount` and `Time` by class

Transaction amount usually has a long-tailed distribution. We use `log1p(Amount)` for visualization, which means `log(1 + Amount)` and preserves zero amounts. The histograms below normalize each class separately so the rare fraud distribution remains visible; they are not showing the overall class prevalence.

In [ ]:
if "Amount" in df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    for label, name in [(0, "Legitimate (0)"), (1, "Fraud (1)")]:
        values = pd.to_numeric(df.loc[df[TARGET] == label, "Amount"], errors="coerce").dropna()
        values = values[values >= 0]
        ax.hist(np.log1p(values), bins=70, density=True, histtype="step", linewidth=1.7, label=name)
    ax.set_title("Amount distribution by class")
    ax.set_xlabel("log(1 + Amount)")
    ax.set_ylabel("Density within class")
    ax.legend()
    plt.tight_layout()
    plt.show()

    amount_summary = df.groupby(TARGET)["Amount"].describe(percentiles=[0.25, 0.5, 0.75])
    display(amount_summary)

if "Time" in df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    for label, name in [(0, "Legitimate (0)"), (1, "Fraud (1)")]:
        hours = pd.to_numeric(df.loc[df[TARGET] == label, "Time"], errors="coerce").dropna() / 3600
        ax.hist(hours, bins=60, density=True, histtype="step", linewidth=1.7, label=name)
    ax.set_title("Relative time distribution by class")
    ax.set_xlabel("Time from start of dataset (hours; not clock time)")
    ax.set_ylabel("Density within class")
    ax.legend()
    plt.tight_layout()
    plt.show()

    time_summary = df.groupby(TARGET)["Time"].describe(percentiles=[0.25, 0.5, 0.75])
    display(time_summary)

## 6. Correlation analysis

Pearson correlation measures linear association. Because `Class` is binary, its correlation with each numeric feature gives a simple descriptive ranking. It is **not** a causal claim, and it is not a substitute for model-based feature importance or validation.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr(numeric_only=True)

if TARGET in corr.columns:
    corr_with_target = corr[TARGET].drop(labels=[TARGET]).dropna()
    corr_with_target = corr_with_target.reindex(corr_with_target.abs().sort_values(ascending=False).index)
    display(corr_with_target.to_frame("correlation_with_Class").head(15))

    ax = corr_with_target.sort_values().plot(kind="barh", figsize=(10, 10),
                                              title="Feature correlation with Class")
    ax.set_xlabel("Pearson correlation")
    plt.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(16, 13))
image = ax.imshow(corr.to_numpy(), aspect="auto", vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(np.arange(len(corr.columns)))
ax.set_yticks(np.arange(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticklabels(corr.index, fontsize=7)
ax.set_title("Correlation matrix across numerical columns")
fig.colorbar(image, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()

## 7. Inspect feature distributions

The anonymized components `V1`–`V28` are not human-readable business fields. For a first comparison, plot the six features with the largest absolute univariate correlation with `Class`. Values are clipped at each class's 1st and 99th percentiles **for visualization only** to prevent a few extremes from compressing the rest of the chart. The underlying DataFrame remains unchanged.

In [ ]:
if TARGET in corr.columns:
    ranked = corr[TARGET].drop(labels=[TARGET]).dropna()
    top_features = ranked.reindex(ranked.abs().sort_values(ascending=False).index).head(6).index.tolist()
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, feature in zip(axes.flat, top_features):
        for label, name in [(0, "Legitimate (0)"), (1, "Fraud (1)")]:
            values = pd.to_numeric(df.loc[df[TARGET] == label, feature], errors="coerce").dropna()
            if values.empty:
                continue
            low, high = values.quantile([0.01, 0.99])
            values = pd.Series(np.clip(values.to_numpy(dtype=float), float(low), float(high)))
            ax.hist(values, bins=40, density=True, histtype="step", linewidth=1.5, label=name)
        ax.set_title(f"{feature} (corr={ranked[feature]:.3f})")
        ax.set_ylabel("Density within class")
        ax.legend(fontsize=8)
    for ax in axes.flat[len(top_features):]:
        ax.set_visible(False)
    fig.suptitle("Six features with largest absolute correlation with Class", y=1.01)
    plt.tight_layout()
    plt.show()

## 8. Export a repeatable EDA report

The notebook is for learning and investigation. For repeatable analysis from PowerShell, the companion script `src/data/eda.py` exports a summary JSON, CSV tables, and PNG charts under `reports/eda/`.

After inspecting the notebook above, run this in PowerShell from the repository root:

```powershell
python -m src.data.eda
```

Do not write conclusions until you have examined the actual outputs from your downloaded CSV. Pay special attention to the duplicate counts and any missing values.

In [ ]:
# Optional: generate the same report artifacts from this notebook's already loaded DataFrame.
from src.data.eda import run_eda, resolve_output_dir

output_dir = resolve_output_dir("reports/eda")
eda_summary = run_eda(df, TARGET, expected_columns, output_dir)
print(f"EDA artifacts saved to: {output_dir}")
print(f"Summary JSON: {output_dir / 'eda_summary.json'}")

## 9. Record the findings before modeling

Fill this checklist after running the notebook on the real dataset:

- [ ] Row/column count and expected schema confirmed.
- [ ] Class counts and fraud percentage recorded.
- [ ] Missing values inspected.
- [ ] Exact duplicates and duplicate feature groups inspected; no automatic deletion.
- [ ] Amount and relative-time distributions examined by class.
- [ ] Feature correlations and correlation heatmap reviewed.
- [ ] Limitations recorded: rare fraud, anonymized PCA features, historical data, and `Time` is not a clock timestamp.

**Next phase:** establish a leakage-safe train/validation/test split and implement a Logistic Regression baseline. Do not resample the full dataset before splitting, and keep the held-out test set out of model selection.

## Findings from the uploaded dataset (25 October 2026)

The EDA script was run against the uploaded `creditcard.csv`. Its measured results are:

- **284,807 rows × 31 columns**, with the expected schema and all columns numeric.
- **0 missing values**.
- **284,315 legitimate** transactions and **492 fraud** transactions; fraud is **0.1727%** of the data.
- **1,081 exact duplicate extra copies** across 773 repeated full-row groups; no identical feature groups have conflicting labels. Do not silently delete duplicates.
- `Amount` is right-skewed: median `22.00` for legitimate rows and `9.25` for fraud rows, while means are `88.29` and `122.21` respectively.
- Largest absolute univariate correlations with `Class`: `V17` (-0.3265), `V14` (-0.3025), `V12` (-0.2606), and `V10` (-0.2169). `Amount` (0.0056) and `Time` (-0.0123) have near-zero linear correlations with the target.

These observations are descriptive, not causal. In Phase 3, we will preserve the raw CSV, choose a time-aware split that keeps equal timestamps together, and evaluate models using recall, precision, F1, PR-AUC, ROC-AUC, and the confusion matrix—not accuracy alone. See `reports/eda/PHASE2_FINDINGS.md` for the complete report and `reports/eda/` for generated tables and plots.
